# 04 -- OWASP Python

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- identifier les 10 risques OWASP 2021 en contexte Python ;
- prevenir les injections SQL, NoSQL, commande, template ;
- gerer l'authentification et les sessions de facon sure ;
- proteger contre XSS et CSRF dans Django/Flask/FastAPI ;
- valider les entrees avec Pydantic ;
- auditer vos dependances avec `pip-audit` ;
- utiliser `bandit` pour l'analyse statique de securite ;
- chiffrer les secrets et utiliser un coffre.

## Prerequis -- ce que vous devez maitriser

- fonctions typees ;
- les bases HTTP/HTML ;
- un framework web (Flask ou FastAPI).

Ce que ce notebook **ne couvrira pas** :

- le pentest/reverse ;
- la securite reseau (WAF, IDS) ;
- la conformite (RGPD, ISO 27001).

## Plan

1. OWASP Top 10 2021 (survol)
2. A01 - Broken Access Control
3. A02 - Cryptographic Failures
4. A03 - Injection
5. A04 - Insecure Design
6. A05 - Security Misconfiguration
7. A06 - Vulnerable Components
8. A07 - Auth Failures
9. A08 - Integrity Failures
10. A09 - Logging Failures
11. A10 - SSRF
12. Validation avec Pydantic
13. Bandit et analyse statique
14. pip-audit et dependances
15. Gestion des secrets
16. Synthese
17. Exercices

---

## 1. OWASP Top 10 2021 (survol)

| ID | Nom |
|---|---|
| A01 | Broken Access Control |
| A02 | Cryptographic Failures |
| A03 | Injection |
| A04 | Insecure Design |
| A05 | Security Misconfiguration |
| A06 | Vulnerable and Outdated Components |
| A07 | Identification and Authentication Failures |
| A08 | Software and Data Integrity Failures |
| A09 | Security Logging and Monitoring Failures |
| A10 | Server-Side Request Forgery (SSRF) |

---

## 2. A01 - Broken Access Control

In [ ]:
# MAUVAIS : pas de verif d'ownership
@api.get('/invoices/{invoice_id}')  # noqa: F821
def get_invoice_bad(invoice_id: int, user):
    return db.get(invoice_id)  # noqa: F821  # n'importe qui voit n'importe quoi

In [ ]:
# BON : verif ownership
def get_invoice(invoice_id: int, user):
    inv = db.get(invoice_id)  # noqa: F821
    if inv.owner_id != user.id:
        raise HTTPException(403)  # noqa: F821
    return inv
print('Toujours verifier que l\'user a acces a la ressource')

---

## 3. A02 - Cryptographic Failures

**Erreurs classiques** :

- MD5 ou SHA-1 pour hasher des mots de passe
- ECB au lieu de GCM/CBC
- Cle codee en dur dans le code
- Nonces reutilises

**Bonnes pratiques** :

- `bcrypt`/`argon2`/`scrypt` pour les mots de passe
- `AES-GCM` pour le chiffrement symetrique
- `secrets.token_*` pour les random

In [ ]:
# Hachage de mot de passe (pip install argon2-cffi)
from argon2 import PasswordHasher
ph = PasswordHasher()
hashed = ph.hash('motdepasse')
print(hashed[:50])
ph.verify(hashed, 'motdepasse')

---

## 4. A03 - Injection

### SQL Injection

In [ ]:
# MAUVAIS
user_input = "1 OR 1=1"
query = f"SELECT * FROM users WHERE id = {user_input}"
print('Dangereux :', query)

In [ ]:
# BON : parametres lies
from sqlalchemy import text, create_engine
engine = create_engine('sqlite:///:memory:')
with engine.connect() as conn:
    r = conn.execute(text('SELECT :x AS n'), {'x': user_input})
    print(list(r))

### Injection de commande

In [ ]:
import subprocess
# MAUVAIS : shell=True avec input utilisateur
# subprocess.run(f'ls {user_input}', shell=True)

# BON : liste d'args, jamais shell=True
subprocess.run(['ls', '/tmp'], check=True, capture_output=True)

### Injection de template (SSTI)

In [ ]:
# MAUVAIS : injecter l'input directement dans un template Jinja2
from jinja2 import Template
# Template('Hello ' + user_input).render()  # danger si user_input = '{{7*7}}'

# BON : passer en variable
Template('Hello {{ name }}').render(name=user_input)

---

## 5. A04 - Insecure Design

Exemples :

- Mot de passe recuperable en clair (par mail) -> stocker hashe uniquement
- Pas de limite sur les tentatives de login -> rate limiting
- Pas de MFA pour operations sensibles
- Confiance dans le client (verifications cote serveur absentes)

---

## 6. A05 - Security Misconfiguration

- `DEBUG = True` en production (Django, Flask) : expose stacktraces
- CORS trop permissif (`*`)
- Headers manquants : `X-Content-Type-Options`, `X-Frame-Options`, `Content-Security-Policy`
- TLS faible (TLS 1.0/1.1, RC4)
- Default credentials (admin/admin)

In [ ]:
# Headers securises avec Flask
from flask import Flask
app = Flask(__name__)

@app.after_request
def add_headers(response):
    response.headers['X-Content-Type-Options'] = 'nosniff'
    response.headers['X-Frame-Options'] = 'DENY'
    response.headers['Content-Security-Policy'] = "default-src 'self'"
    return response
print('Headers securises ajoutes')

---

## 7. A06 - Vulnerable Components

In [ ]:
# pip install pip-audit
# pip-audit
import subprocess
print('pip-audit scanne requirements.txt contre la BD OSV')

### Outils

- **pip-audit** (PyPA) : scan des dependances Python
- **safety** : alternative historique
- **dependabot** / **renovate** : MAJ auto GitHub/GitLab
- **Snyk** : commercial avec CVE + fixes

---

## 8. A07 - Authentication Failures

- Mot de passe faible (pas de politique)
- Pas de limite de tentatives
- Stockage en clair ou MD5
- Sessions non invalidees au logout
- Tokens predictibles
- Pas de MFA

In [ ]:
import secrets
# Token aleatoire securise (jamais random.random())
session_token = secrets.token_urlsafe(32)
print(session_token)

---

## 9. A08 - Integrity Failures

- CI/CD non signee : push malicieux accepte
- Packages non verifies (`pip install <url>`)
- Images docker pulled sans signature (cosign)
- Deserialisation non-sure (**pickle** de donnees externes)

In [ ]:
import pickle
# NE JAMAIS faire pickle.loads(bytes_from_internet)
# Equivalent d'executer du code arbitraire
print('pickle est NON sur pour donnees externes')

---

## 10. A09 - Logging Failures

In [ ]:
import logging
logger = logging.getLogger('app')

# Ne JAMAIS logger les mots de passe
# logger.info('Login avec %s', password)  # MAUVAIS

# Logger les evenements de securite
logger.info('login_success user_id=%s ip=%s', 42, '1.2.3.4')
print('Bien logger sans secrets')

---

## 11. A10 - SSRF

In [ ]:
import httpx

# MAUVAIS : fetch une URL fournie par l'utilisateur sans controle
# r = httpx.get(user_url)  # danger : user_url='http://169.254.169.254/latest/meta-data'

# BON : allowlist de domaines
ALLOWED = {'example.com', 'api.example.com'}
from urllib.parse import urlparse

def safe_fetch(url: str) -> str:
    host = urlparse(url).hostname
    if host not in ALLOWED:
        raise ValueError('host interdit')
    return 'ok'

safe_fetch('https://example.com/data')

---

## 12. Validation avec Pydantic

In [ ]:
from pydantic import BaseModel, EmailStr, Field, ValidationError

class UserIn(BaseModel):
    email: EmailStr
    age: int = Field(ge=0, le=150)
    name: str = Field(min_length=1, max_length=100)

try:
    UserIn(email='pas-un-email', age=-1, name='')
except ValidationError as e:
    print(len(e.errors()), 'erreurs detectees')

---

## 13. Bandit et analyse statique

In [ ]:
# pip install bandit
# bandit -r mon_projet/
print('Bandit detecte : hardcoded passwords, pickle, subprocess shell=True, SQL sans params, ...')

### Plugins complementaires

- **ruff** : lint avec regles de securite `S` (equivalentes bandit)
- **semgrep** : patterns de securite custom
- **mypy strict** : catche des bugs de type qui cachent des failles

---

## 14. pip-audit et dependances

In [ ]:
# Verifier le pinning
# pip freeze > requirements.txt
# pip-audit -r requirements.txt
print('Exemples de vuln : jinja2 < 3.1.4, requests < 2.32, ...')

---

## 15. Gestion des secrets

**Jamais dans le code** :

- `.env` + `python-dotenv` (local)
- Variables d'environnement (12-factor)
- Coffres : **HashiCorp Vault**, **AWS Secrets Manager**, **Doppler**, **Infisical**
- **systemd credentials** pour les services Linux

**Detection de secrets** : `detect-secrets`, `gitleaks`, `truffleHog`.

In [ ]:
import os
# os.environ['DATABASE_URL'] lu depuis .env ou l'environnement
print('Jamais hardcoder des credentials')

---

## Synthese

| Risque | Mesure cle |
|---|---|
| A01 Access Control | verif owner, 403 par defaut |
| A02 Crypto | argon2, AES-GCM, `secrets` |
| A03 Injection | params lies, pas de `shell=True` |
| A04 Insecure Design | threat modeling, rate limit, MFA |
| A05 Misconfig | headers securite, TLS 1.2+ |
| A06 Vulnerable deps | pip-audit + dependabot |
| A07 Auth | bcrypt/argon2, sessions |
| A08 Integrity | signer les artifacts, pas de pickle |
| A09 Logging | event d'auth, pas de secrets |
| A10 SSRF | allowlist, pas de fetch user url |

### Outils

- **ruff** `S` rules = bandit rapide
- **pip-audit** pour dependances
- **detect-secrets** pre-commit hook
- **Pydantic** pour valider tout input externe

---

## Exercices

### Exercice 1 (facile) -- Hasher un mot de passe

Utilisez `argon2` pour hasher le mot de passe `'secret123'` et verifiez-le.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from argon2 import PasswordHasher, exceptions
ph = PasswordHasher()
h = ph.hash('secret123')
print(h)
print('Verif OK :', ph.verify(h, 'secret123'))
try:
    ph.verify(h, 'mauvais')
except exceptions.VerifyMismatchError:
    print('mauvais mdp')
```

</details>

### Exercice 2 (facile) -- Validation Pydantic

Creez un modele `Contact` avec `name` (1-50), `email` valide, `age` entre 18 et 120.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from pydantic import BaseModel, EmailStr, Field

class Contact(BaseModel):
    name: str = Field(min_length=1, max_length=50)
    email: EmailStr
    age: int = Field(ge=18, le=120)

c = Contact(name='Alice', email='alice@example.com', age=30)
print(c)
```

</details>

### Exercice 3 (moyen) -- SSRF guard

Ecrivez une fonction `safe_url(url: str, allowed: set[str]) -> str` qui leve `ValueError` si le host n'est pas dans la liste ou si le scheme n'est pas http/https.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from urllib.parse import urlparse

def safe_url(url: str, allowed: set[str]) -> str:
    parsed = urlparse(url)
    if parsed.scheme not in {'http', 'https'}:
        raise ValueError('scheme non autorise')
    if parsed.hostname not in allowed:
        raise ValueError('host non autorise')
    return url

print(safe_url('https://example.com/x', {'example.com'}))
try:
    safe_url('file:///etc/passwd', {'example.com'})
except ValueError as e:
    print('OK reject :', e)
```

</details>

### Exercice 4 (moyen) -- Safe subprocess

Ecrivez une fonction `safe_ls(path: str) -> str` qui liste un repertoire sans utiliser `shell=True` et en verifiant que `path` est sous `/tmp`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import subprocess
import pathlib

def safe_ls(path: str) -> str:
    p = pathlib.Path(path).resolve()
    if not str(p).startswith('/tmp'):
        raise ValueError('path hors /tmp')
    r = subprocess.run(['ls', '-la', str(p)], capture_output=True, text=True, check=True)
    return r.stdout

print(safe_ls('/tmp'))
```

</details>

### Exercice 5 (difficile) -- Rate limiter

Implementez un `RateLimiter(max_attempts, window_seconds)` qui permet au plus `max_attempts` par fenetre glissante par cle (ex. IP ou user_id). Leve `RateLimitError` sinon.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import time
from collections import defaultdict, deque

class RateLimitError(Exception):
    pass

class RateLimiter:
    def __init__(self, max_attempts: int, window_seconds: float) -> None:
        self.max = max_attempts
        self.window = window_seconds
        self.history: dict[str, deque[float]] = defaultdict(deque)

    def check(self, key: str) -> None:
        now = time.monotonic()
        h = self.history[key]
        while h and now - h[0] > self.window:
            h.popleft()
        if len(h) >= self.max:
            raise RateLimitError(f'Too many attempts for {key}')
        h.append(now)

rl = RateLimiter(3, 60)
for i in range(4):
    try:
        rl.check('ip:1.2.3.4')
        print('OK', i)
    except RateLimitError as e:
        print('REJ', e)
```

</details>

---

## Ressources externes

- [OWASP Top 10 2021](https://owasp.org/Top10/)
- [OWASP Python Security](https://owasp.org/www-project-python-security/)
- [Bandit](https://bandit.readthedocs.io/)
- [pip-audit](https://pypi.org/project/pip-audit/)
- [detect-secrets](https://github.com/Yelp/detect-secrets)